# Superliga Scout — Data Exploration

This notebook covers:
1. Data quality checks for all scraped leagues
2. League coefficient visualisation
3. Style cluster analysis
4. Feature correlation matrix
5. Historical transfer outcome analysis
6. Age curve visualisation
7. Adaptability score distribution

In [ ]:
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sys.path.insert(0, str(Path('.').resolve().parent))

DB_PATH = Path('..') / 'data' / 'processed' / 'scouting.db'

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Notebook ready.')

## 1. Load Data from SQLite

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist()
    print('Tables in DB:', tables)
    
    profiles = pd.read_sql('SELECT * FROM player_profiles', conn) if 'player_profiles' in tables else pd.DataFrame()
    hist_tf  = pd.read_sql('SELECT * FROM historical_transfers', conn) if 'historical_transfers' in tables else pd.DataFrame()
    mkt_vals = pd.read_sql('SELECT * FROM market_values', conn) if 'market_values' in tables else pd.DataFrame()

print(f'player_profiles : {profiles.shape}')
print(f'historical_transfers: {hist_tf.shape}')
print(f'market_values   : {mkt_vals.shape}')

## 2. Data Quality Checks

In [ ]:
if not profiles.empty:
    print('=== player_profiles ===')
    print(profiles.dtypes)
    print('\nNull % per column:')
    null_pct = (profiles.isnull().sum() / len(profiles) * 100).sort_values(ascending=False)
    print(null_pct[null_pct > 0].head(20))

    print('\nLeague distribution:')
    if 'league_key' in profiles.columns:
        print(profiles['league_key'].value_counts())

    print('\nSeason distribution:')
    if 'season' in profiles.columns:
        print(profiles['season'].value_counts().sort_index())

In [ ]:
# Per-league player counts across seasons
if not profiles.empty and 'league_key' in profiles.columns and 'season' in profiles.columns:
    pivot = profiles.groupby(['season', 'league_key']).size().unstack(fill_value=0)
    pivot.plot(kind='bar', figsize=(14, 5), stacked=False)
    plt.title('Player records per season and league')
    plt.xlabel('Season')
    plt.ylabel('Player-season records')
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()

## 3. League Coefficient Visualisation

In [ ]:
from src.features import (
    compute_league_coefficients,
    _default_league_coefficients,
)

# Calibrated from data (if available) or defaults
if not hist_tf.empty:
    coeffs = compute_league_coefficients(hist_tf, profiles)
else:
    coeffs = _default_league_coefficients()
    print('Using default league coefficients (no historical transfers yet).')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
leagues = sorted(coeffs.items(), key=lambda x: x[1])
names  = [l[0].replace('_', ' ').title() for l in leagues]
values = [l[1] for l in leagues]
colors = ['#C8102E' if n.lower() == 'superliga' else '#0D1B40' for n in names]

bars = ax.barh(names, values, color=colors)
ax.axvline(1.0, color='#E8A000', linewidth=2, linestyle='--', label='Superliga baseline')
ax.set_xlabel('League Coefficient (relative to Danish Superliga)')
ax.set_title('Eastern European League Quality Coefficients')
ax.legend()

for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Style Cluster Analysis

In [ ]:
from src.features import build_style_clusters, STYLE_LABELS

if not profiles.empty:
    df_clustered, kmeans, scaler = build_style_clusters(profiles)

    # Cluster size bar chart
    counts = df_clustered['style_cluster'].value_counts().sort_index()
    labels = [STYLE_LABELS.get(i, str(i)) for i in counts.index]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Cluster sizes
    axes[0].barh(labels, counts.values, color='#0D1B40')
    axes[0].set_title('Style Cluster Sizes')
    axes[0].set_xlabel('Player-season records')
    
    # Cluster distribution by league
    if 'league_key' in df_clustered.columns:
        cross = pd.crosstab(
            df_clustered['league_key'],
            df_clustered['style_label'],
            normalize='index'
        )
        cross.plot(kind='bar', stacked=True, ax=axes[1], colormap='tab10')
        axes[1].set_title('Style Distribution per League')
        axes[1].set_xlabel('League')
        axes[1].set_ylabel('Proportion')
        axes[1].legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)
        axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print('No profile data loaded.')

## 5. Feature Correlation Matrix

In [ ]:
CORE_METRICS = [
    'standard_xg', 'standard_xa', 'standard_gls', 'standard_ast',
    'goal_shot_creation_sca90',
    'passing_prog_p', 'passing_xa',
    'possession_prg_c', 'possession_touches',
    'defense_tkl', 'defense_press', 'defense_int',
]

if not profiles.empty:
    available = [c for c in CORE_METRICS if c in profiles.columns]
    if available:
        corr = profiles[available].corr()
        fig, ax = plt.subplots(figsize=(10, 8))
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
                    center=0, square=True, linewidths=0.5, ax=ax, fontsize=7)
        ax.set_title('Feature Correlation Matrix (per-90 stats)')
        plt.tight_layout()
        plt.show()
    else:
        print('Core metric columns not found in profiles.')

## 6. Historical Transfer Outcome Analysis

In [ ]:
if not hist_tf.empty and 'target_performance_pct' in hist_tf.columns:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Distribution of outcomes
    axes[0].hist(hist_tf['target_performance_pct'].dropna(), bins=20, color='#0D1B40', edgecolor='white')
    axes[0].axvline(50, color='#C8102E', linestyle='--', label='Median (50th pct)')
    axes[0].set_title('Superliga Performance Distribution\n(post-transfer)')
    axes[0].set_xlabel('Performance percentile')
    axes[0].legend()
    
    # By origin league
    if 'league_key' in hist_tf.columns:
        hist_tf.boxplot(
            column='target_performance_pct',
            by='league_key',
            ax=axes[1]
        )
        axes[1].set_title('Outcome by Origin League')
        axes[1].set_xlabel('')
        axes[1].tick_params(axis='x', rotation=45)
    
    # By style cluster (if computed)
    if 'style_cluster' in hist_tf.columns:
        hist_tf.boxplot(
            column='target_performance_pct',
            by='style_cluster',
            ax=axes[2]
        )
        axes[2].set_title('Outcome by Style Cluster')
        axes[2].set_xlabel('Cluster ID')
    
    plt.suptitle('')
    plt.tight_layout()
    plt.show()
else:
    print('No historical transfer data available yet.')
    print('This table will populate once the full pipeline is run.')

## 7. Age Curve Visualisation

In [ ]:
from src.features import compute_age_curve_factor

ages = np.arange(16, 38, 0.5)
positions = ['FW', 'MF', 'DF', 'GK']
colors_map = {'FW': '#C8102E', 'MF': '#0D1B40', 'DF': '#2ECC71', 'GK': '#E8A000'}

fig, ax = plt.subplots(figsize=(10, 5))
for pos in positions:
    factors = [compute_age_curve_factor(a, pos) for a in ages]
    ax.plot(ages, factors, label=pos, color=colors_map[pos], linewidth=2)

ax.axhline(1.0, color='grey', linestyle=':', linewidth=1)
ax.fill_between(ages, 0.95, 1.0, alpha=0.08, color='green', label='Peak zone')
ax.set_xlabel('Age')
ax.set_ylabel('Performance multiplier')
ax.set_title('Age-Performance Curve by Position\n(Peak: FW ~25, MF ~26, DF/GK ~27-29)')
ax.legend()
ax.set_ylim(0.65, 1.05)
ax.set_xlim(16, 37)
plt.tight_layout()
plt.show()

## 8. Adaptability Score Distribution

In [ ]:
if not profiles.empty and 'adaptability_score' in profiles.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Overall distribution
    axes[0].hist(profiles['adaptability_score'].dropna(), bins=30,
                 color='#0D1B40', edgecolor='white')
    axes[0].set_xlabel('Adaptability Score (0-10)')
    axes[0].set_ylabel('Players')
    axes[0].set_title('Adaptability Score Distribution')
    axes[0].axvline(7, color='#2ECC71', linestyle='--', label='High threshold (7)')
    axes[0].axvline(4, color='#C8102E', linestyle='--', label='Low threshold (4)')
    axes[0].legend(fontsize=8)
    
    # By league
    if 'league_key' in profiles.columns:
        profiles.boxplot(column='adaptability_score', by='league_key', ax=axes[1])
        axes[1].set_title('Adaptability by League')
        axes[1].set_xlabel('')
        axes[1].tick_params(axis='x', rotation=45)
    
    plt.suptitle('')
    plt.tight_layout()
    plt.show()
    
    # Summary stats
    print('Adaptability score summary:')
    print(profiles['adaptability_score'].describe())
else:
    print('No adaptability_score column found.')
    print('Run src.features.compute_adaptability_score() first.')

## 9. Top Candidates Preview (from Predictions)

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    tbls = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist()
    if 'predictions' in tbls:
        preds = pd.read_sql('SELECT * FROM predictions ORDER BY predicted_pct DESC', conn)
        print(f'Top 10 candidates:')
        display_cols = [c for c in [
            'player', 'league_key', 'predicted_pct', 'ci_lower', 'ci_upper',
            'adaptability_score', 'style_label', 'risk_rating', 'estimated_market_value_eur'
        ] if c in preds.columns]
        print(preds[display_cols].head(10).to_string(index=False))
    else:
        print('No predictions table found. Run python -m src.model first.')